# 🗂️ Notebook 2: Google Search — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/google-search
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Key data structures

### Inverted index (term → postings list)
```
"python"   → [ (doc=42, tf=3, positions=[12,45,98]),
               (doc=99, tf=1, positions=[5]),
               ... ]
"flask"    → [ (doc=42, tf=1, positions=[58]), ... ]
```

### Forward index (doc → metadata)
```
doc=42 → { url, title, length, pagerank, crawl_ts }
```

### Simple query flow (intersection)
For `python flask`:
1. Look up postings for `python`.
2. Look up postings for `flask`.
3. **Intersect** doc-ids.
4. Score each by TF-IDF + PageRank + freshness + user signals.
5. Return top-K.


## APIs (internal)

```http
# Public
GET /search?q=python+flask&n=10

# Internal — between query shard + aggregator
POST /shard/query       { q, n, filters }   → [ { doc_id, score } ]
POST /doc-fetch         { doc_ids: [...] }  → [ { title, snippet, url } ]
```

The *aggregator* sends the query to all shards in parallel, then merges results by score.


In [ ]:
# Build a tiny inverted index + run a query — end to end in a few lines.
from collections import defaultdict
import re, math

docs = {
    1: "Python is a programming language",
    2: "Flask is a web framework for Python",
    3: "Django is another Python web framework",
    4: "Rust is a systems programming language",
}

def tokens(text): return re.findall(r"[a-z]+", text.lower())

# term -> {doc_id: term_freq}
index: dict[str, dict[int,int]] = defaultdict(dict)
for doc_id, text in docs.items():
    for w in tokens(text):
        index[w][doc_id] = index[w].get(doc_id, 0) + 1

N = len(docs)
def idf(term): return math.log(N / (1 + len(index.get(term, {}))))

def search(q, k=3):
    qs = tokens(q)
    # intersect postings
    postings = [set(index.get(t, {}).keys()) for t in qs]
    if not postings: return []
    candidates = set.intersection(*postings) if all(postings) else set()
    # score = sum over query-terms of tf-idf
    scored = []
    for d in candidates:
        s = sum(index[t][d] * idf(t) for t in qs)
        scored.append((d, round(s, 3), docs[d]))
    return sorted(scored, key=lambda x: -x[1])[:k]

print("python web framework →", search("python web framework"))
print("programming language →", search("programming language"))


## Bad -> Best: how the retrieval step evolves

We'll build the same query four times, each better than the previous.
Run every cell and watch the results sharpen.


In [ ]:
# Shared toy corpus used by all the versions below.
DOCS = {
    1: 'Python is a popular programming language used for web, data, and scripting',
    2: 'Flask is a lightweight Python web framework with a small core',
    3: 'Django is a batteries-included Python web framework for large apps',
    4: 'Rust is a fast systems programming language loved for its safety',
    5: 'JavaScript is the programming language of the web browser',
    6: 'The Python Package Index hosts thousands of Python libraries',
}
QUERY = 'python web framework'


### Bad - linear scan (grep the whole corpus per query)

For every query we touch every document. Works for 6 docs.
It does not work for 50 *billion*.


In [ ]:
import re, time

def tok(t): return re.findall(r'[a-z]+', t.lower())

def linear_search(query):
    qs = set(tok(query))
    hits = []
    for doc_id, text in DOCS.items():
        ws = set(tok(text))
        if qs.issubset(ws):
            hits.append(doc_id)
    return hits

t0 = time.perf_counter()
print(linear_search(QUERY), 'in', round((time.perf_counter()-t0)*1e6,1), 'us')
# Cost: O(#docs * avg-doc-length) per query.


### Better - inverted index

Flip the loop: for each **term** pre-compute the list of docs that contain it (the
*postings list*). At query time we only touch docs that contain at least one query term.


In [ ]:
from collections import defaultdict

index = defaultdict(set)           # term -> {doc_id, ...}
for d, text in DOCS.items():
    for w in tok(text):
        index[w].add(d)

def inv_search(query):
    postings = [index.get(t, set()) for t in tok(query)]
    return sorted(set.intersection(*postings)) if all(postings) else []

print(inv_search(QUERY))
# Cost: O(length of shortest postings list). Huge win for rare terms.


### Good - TF-IDF scoring

Intersection tells us *which* docs match. We also need to rank them.
TF-IDF rewards docs with many matches (**TF**), while damping common words (**IDF**).


In [ ]:
import math

tf = defaultdict(lambda: defaultdict(int))   # tf[doc][term]
df = defaultdict(int)                        # df[term] = # docs containing term
for d, text in DOCS.items():
    seen = set()
    for w in tok(text):
        tf[d][w] += 1
        seen.add(w)
    for w in seen:
        df[w] += 1

N = len(DOCS)
def idf(t): return math.log((N + 1) / (df.get(t, 0) + 1)) + 1   # smoothed

def tfidf_search(query, k=3):
    qs = tok(query)
    cand = set.intersection(*(index.get(t, set()) for t in qs)) if qs else set()
    scored = [(d, sum(tf[d][t] * idf(t) for t in qs)) for d in cand]
    return sorted(scored, key=lambda x: -x[1])[:k]

for d, s in tfidf_search(QUERY):
    print(f'{s:5.2f}  doc{d}: {DOCS[d]}')


### Best - BM25 (Okapi BM25)

BM25 is the de-facto standard used by Lucene / Elasticsearch / OpenSearch.
Improvements over TF-IDF:

- **Saturated TF**: the 10th occurrence of a word adds less than the 2nd (parameter `k1`).
- **Length normalization**: short pages that contain the word are not dwarfed by long pages (parameter `b`).


In [ ]:
doc_len = {d: len(tok(t)) for d, t in DOCS.items()}
avg_len = sum(doc_len.values()) / len(doc_len)

def bm25(query, k=3, k1=1.5, b=0.75):
    qs = tok(query)
    cand = set.intersection(*(index.get(t, set()) for t in qs)) if qs else set()
    scored = []
    for d in cand:
        s = 0.0
        for t in qs:
            f   = tf[d][t]
            i   = idf(t)
            norm = 1 - b + b * doc_len[d] / avg_len
            s += i * (f * (k1 + 1)) / (f + k1 * norm)
        scored.append((d, s))
    return sorted(scored, key=lambda x: -x[1])[:k]

for d, s in bm25(QUERY):
    print(f'{s:5.2f}  doc{d}: {DOCS[d]}')


#### Why BM25 wins in practice
- Tune `k1` (term-frequency saturation, typical 1.2-2.0) and `b` (length penalty, ~0.75).
- Handles spammy pages better (a word repeated 1000 times does not dominate).
- Same data structures as TF-IDF - it is a **drop-in upgrade**.


## Tokenization matters more than scoring

Before any scoring, the text has to be turned into tokens. A real pipeline does:

1. **Lowercase** so `Python` and `python` match.
2. **Stop-word removal** - `the`, `is`, `a` carry no signal and cost postings-list space.
3. **Stemming** - `run`, `running`, `runs` all map to `run`.
4. **Unicode normalization** - accents and quotes folded to a canonical form.

Here is a tiny, dependency-free version.


In [ ]:
STOP = {'a','an','the','is','are','of','for','and','or','to','in','on','with','its','used'}

def stem(w):
    # Poor-mans Porter stemmer. Strip common suffixes and collapse
    # a trailing double consonant left behind (e.g. 'running' -> 'runn' -> 'run').
    for suf in ('ings', 'ing', 'ies', 'es', 'ed', 'ly', 's'):
        if len(w) > len(suf) + 2 and w.endswith(suf):
            w = w[:-len(suf)]
            break
    if len(w) > 2 and w[-1] == w[-2] and w[-1] not in 'aeiou':
        w = w[:-1]
    return w

def pipeline(text):
    return [stem(w) for w in tok(text) if w not in STOP]

print(pipeline('Running a lightweight Python framework for the web'))
# expected: ['run', 'lightweight', 'python', 'framework', 'web']


## Boolean operators and phrase queries

Real search engines accept `python AND (flask OR django)` and `"web framework"` exactly.
Postings lists make these cheap:

- `AND` -> set intersection
- `OR`  -> set union
- `NOT` -> set difference
- `"phrase"` -> intersect, then check **positions** are consecutive


In [ ]:
# Positional index: term -> {doc_id: [positions]}
positional = defaultdict(lambda: defaultdict(list))
for d, text in DOCS.items():
    for pos, w in enumerate(tok(text)):
        positional[w][d].append(pos)

def phrase_search(phrase):
    words = tok(phrase)
    if not words: return []
    cand = set.intersection(*(set(positional[w]) for w in words))
    hits = []
    for d in cand:
        first_positions = positional[words[0]][d]
        for p0 in first_positions:
            if all((p0 + i) in positional[words[i]][d] for i in range(1, len(words))):
                hits.append(d); break
    return hits

print('"web framework"      ->', phrase_search('web framework'))
print('"programming language" ->', phrase_search('programming language'))


## Snippet generation

The SERP (search-engine-results page) shows a short excerpt with query terms highlighted.
Pick the window with the highest query-term density.


In [ ]:
def snippet(text, query, window=8):
    words = text.split()
    qs = set(tok(query))
    best_i, best_hits = 0, -1
    for i in range(len(words)):
        w = words[i:i+window]
        hits = sum(1 for x in w if x.lower().strip('.,') in qs)
        if hits > best_hits:
            best_hits, best_i = hits, i
    w = words[best_i:best_i+window]
    return ' '.join(f'<b>{x}</b>' if x.lower().strip('.,') in qs else x for x in w)

print(snippet(DOCS[3], 'python web framework'))
